# Find duplicate images

The vision version of train/valid leakage, and the reason a model can score 0.95
on your split and 0.6 on anyone else's. Scraped and re-exported datasets are
full of the same photo twice — sometimes resized, sometimes recompressed, which
is why a byte comparison finds nothing.

In [ ]:
import orinth

DATASET_ID = next(
    dataset.id
    for dataset in orinth.datasets.list()
    if dataset.task_type in {"classification", "object_detection", "segmentation"}
    and dataset.total_items > 20
)
ref = orinth.datasets.get(DATASET_ID)
print(ref.name, ref.splits)

## Average hash

64 bits per image: shrink to 8×8 greyscale, then one bit per pixel for
above-or-below the mean. It survives resizing and recompression — which is the
whole point — and it is a few lines rather than a dependency.

In [ ]:
import numpy as np
from PIL import Image


def average_hash(path) -> int:
    with Image.open(path) as handle:
        pixels = np.asarray(handle.convert("L").resize((8, 8)), dtype="float32")
    bits = (pixels > pixels.mean()).flatten()
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return value


hashes = []
for split in ("train", "valid", "test"):
    for path in orinth.datasets.paths(DATASET_ID, split)[:400]:
        hashes.append({"split": split, "item": path.name, "hash": average_hash(path)})

print(len(hashes), "images hashed")

## Exact collisions

Same hash means visually identical at 8×8 — near enough that a model cannot use
one to learn about the other.

In [ ]:
import polars as pl

frame = pl.DataFrame(hashes)
groups = frame.group_by("hash").agg(
    pl.len().alias("copies"),
    pl.col("split").unique().alias("splits"),
    pl.col("item").alias("items"),
).filter(pl.col("copies") > 1).sort("copies", descending=True)

print(f"{groups.height} duplicate groups covering {groups['copies'].sum()} images")
groups.head(10)

## The ones that leak

A duplicate inside `train` costs you a little sample weight. A duplicate that
spans `train` and `valid` costs you the validity of the whole split — the model
has seen the answer.

In [ ]:
leaking = groups.filter(pl.col("splits").list.len() > 1)
print(f"{leaking.height} groups span more than one split")
if leaking.height:
    print(leaking.select(["copies", "splits", "items"]).head(10))
    print("\nRemove one side and re-split, or these numbers are not measuring generalisation.")
else:
    print("No cross-split duplicates — the split is honest on this measure.")

## Near-duplicates

Hamming distance over the hashes. Under about 5 bits is the same picture with a
different crop or filter; this is deliberately capped at a sample because the
comparison is quadratic and the answer does not need every pair.

In [ ]:
sample = frame.head(250)
values = sample["hash"].to_list()
items = sample["item"].to_list()

near = []
for i in range(len(values)):
    for j in range(i + 1, len(values)):
        distance = bin(values[i] ^ values[j]).count("1")
        if 0 < distance <= 5:
            near.append({"a": items[i], "b": items[j], "bits_apart": distance})

print(len(near), "near-duplicate pairs in a", len(values), "image sample")
pl.DataFrame(near).sort("bits_apart").head(10) if near else print("none")